<a href="https://colab.research.google.com/github/rit1217/Study-RAG/blob/gemini_file_search/introduction_to_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set-up  

In [4]:
from google.colab import drive, userdata
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google import genai
from google.genai import types
import time

import pandas as pd

# Gemini Client Initiation

In [31]:
client = genai.Client(api_key=userdata.get('gemini_api_key'))

# File name will be visible in citations
file_search_store = client.file_search_stores.create(
    config={'display_name': 'bot_policy_20240702'
    })

# Add File To File Search Store

In [29]:
def store_file(cilent, file_name, file_path='/content/drive/MyDrive/KKP/'):
  operation = client.file_search_stores.upload_to_file_search_store(
    file= f"{file_path}{file_name}",
    file_search_store_name=file_search_store.name,
    config={
        'display_name' : file_name,
    }
  )
  return operation

In [32]:
operation = store_file(client, '20240702_BOT_Policy.pdf')

while not operation.done:
    time.sleep(5)
    operation = client.operations.get(operation)


In [33]:
# List all file search stores
for fs in client.file_search_stores.list():
    print(f" {file_search_store.name} - {file_search_store.display_name}")

 fileSearchStores/botpolicy20240702-w2uusfol5kgo - bot_policy_20240702
 fileSearchStores/botpolicy20240702-w2uusfol5kgo - bot_policy_20240702


In [ ]:
# client.file_search_stores.delete(name='fileSearchStores/botpolicy20240702-2ot0jq8eztge', config={'force':True})

# Gemini Test Prompt

In [34]:
question="""1. ที่ดินที่ได้มาจากการที่บิดายกให้บุตรโดยเสน่หา บิดาซึ่งเป็นผู้ให้ยังสามารถเรียกคืนจากบุตรได้หรือไม่ เรียกคืนจากสาเหตุใดได้บ้าง
2. กรณีที่บุตรนำที่ดินไปดำเนินการจัดสรรแล้ว ยังจะสามารถถูกเรียกคืนการให้จากผู้ให้ได้หรือไม่"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"""คุณเป็นผู้เชี่ยวชาญทางกฎหมายของธนาคารเอกชนแห่งหนึ่ง และตอบคำถามต่อไปนี้อย่างรอบคอบและกระชับ ไม่ยาวจนเกินไป\n{question}""",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print(response.text)

ขอเรียนชี้แจงดังนี้ครับ:

1.  **ที่ดินที่บิดายกให้บุตรโดยเสน่หา บิดาสามารถเรียกคืนได้หรือไม่ และจากสาเหตุใดบ้าง**
    โดยหลักแล้ว การให้โดยเสน่หาเมื่อสมบูรณ์แล้ว ผู้ให้ไม่สามารถเรียกคืนได้ เว้นแต่จะมีเหตุตามกฎหมายกำหนดไว้เป็นการเฉพาะ ซึ่งตามประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 531 และ 532 บิดา (ผู้ให้) สามารถเรียกคืนการให้จากบุตร (ผู้รับ) ได้ในกรณีดังต่อไปนี้ครับ
    *   **การประพฤติเนรคุณ:** หากบุตรประพฤติเนรคุณต่อบิดาอย่างร้ายแรง ซึ่งรวมถึงกรณีดังต่อไปนี้
        *   บุตรได้กระทำความผิดอาญาอย่างร้ายแรงต่อบิดา ไม่ว่าจะเป็นต่อร่างกาย ทรัพย์สิน หรือเสรีภาพของบิดา
        *   บุตรได้ทำให้บิดาเสื่อมเสียชื่อเสียง หรือหมิ่นประมาทบิดาอย่างร้ายแรง
        *   บุตรได้บอกปัดไม่ยอมให้สิ่งของจำเป็นเลี้ยงชีวิตแก่บิดา ในเวลาที่บิดากำลังยากไร้และบุตรสามารถจะให้ได้
    *   **การไม่ชำระหนี้สินของผู้ให้:** ในกรณีที่การให้นั้นมีภาระติดพัน และบุตรไม่ปฏิบัติตามภาระนั้น บิดาอาจเรียกคืนการให้ได้

2.  **กรณีที่บุตรนำที่ดินไปดำเนินการจัดสรรแล้ว ยังจะสามารถถูกเรียกคืนการให้จากผู้ให้ได้หรือไม่**
    หากบุตรได้นำที่